# Flow Field Demo

This notebook shows how a smooth random field becomes a field of angles, and how tracing lines along those angles builds the flow-field image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from math_vis import smooth_field, sample, make_cmap, finish, fig_to_img

# ---------- parameters ----------
SEED    = 11
BG      = '#14161f'
N_LINES = 4000      # lower for the explainer; raise for final print
STEPS   = 200
STEP    = 0.0035
SMOOTH  = 22
TWIST   = 0.9
DEMO_LINES = 150

rng = np.random.default_rng(SEED)
palette = make_cmap(['#1d3557', '#3a6ea5', '#5bc0be', '#f4e3b1', '#f2a65a', '#d1495b'])

In [ ]:
# ---------- BLOCK 1: smooth random field ----------
GW, GH = 256, 320
field = smooth_field((GH, GW), SMOOTH, rng)
angle = field * np.pi * TWIST

def at(f, pts):
    return sample(f, pts[:, 1] * GW, pts[:, 0] * GW)

H = GH / GW
print('field shape:', field.shape, 'canvas height:', H)

In [ ]:
# ---------- BLOCKS 2-4: seed, integrate, color ----------
side = int(np.sqrt(N_LINES))
gx, gy = np.meshgrid(np.linspace(-0.05, 1.05, side), np.linspace(-0.05, 1.30, N_LINES // side))
pts = np.column_stack([gx.ravel(), gy.ravel()])
pts = pts + rng.normal(0, 0.006, pts.shape)
n = len(pts)
seeds = pts.copy()

paths = np.empty((n, STEPS, 2))
paths[:, 0] = pts
for i in range(1, STEPS):
    a = at(angle, pts)
    pts = pts + STEP * np.column_stack([np.cos(a), np.sin(a)])
    paths[:, i] = pts

start_value = at(field, paths[:, 0])
t = 0.5 + 0.28 * start_value + rng.normal(0, 0.05, n)
colors = palette(np.clip(t, 0, 1))
colors[:, 3] = rng.uniform(0.15, 0.5, n)
widths = rng.uniform(0.3, 1.4, n)

In [ ]:
# ---------- EXPLAINER: six panels ----------
def canvas(ax, title):
    ax.set_facecolor(BG)
    ax.set_xlim(0, 1); ax.set_ylim(0, H); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_title(title, color='#e6e6e6', fontsize=11)

demo = rng.choice(n, DEMO_LINES, replace=False)
fig, axs = plt.subplots(2, 3, figsize=(15, 12.4), facecolor=BG)

ax = axs[0,0]; canvas(ax, '1. Smooth random field')
ax.imshow(field, origin='lower', extent=(0,1,0,H), cmap=palette, vmin=-2.5, vmax=2.5)

ax = axs[0,1]; canvas(ax, '2. Field → angle')
ax.imshow(np.mod(angle, 2*np.pi), origin='lower', extent=(0,1,0,H), cmap='twilight', vmin=0, vmax=2*np.pi)
r_, c_ = np.arange(8, GH, 18), np.arange(8, GW, 18)
X, Y = np.meshgrid((c_+0.5)/GW, (r_+0.5)/GW)
a_ = angle[np.ix_(r_, c_)]
ax.quiver(X, Y, np.cos(a_), np.sin(a_), color='white', pivot='mid', scale=26, width=0.004)

ax = axs[0,2]; canvas(ax, '3. Seed points')
ax.scatter(seeds[:,0], seeds[:,1], s=1.5, color='#f4e3b1', lw=0)

ax = axs[1,0]; canvas(ax, '4. Integrate along the wind')
ax.imshow(field, origin='lower', extent=(0,1,0,H), cmap='Greys_r', vmin=-4, vmax=4, alpha=0.25)
ax.add_collection(LineCollection(paths[demo], colors='#f4e3b1', linewidths=0.8))
ax.scatter(seeds[demo,0], seeds[demo,1], s=8, color='#d1495b', zorder=3)

ax = axs[1,1]; canvas(ax, '5. Color per line')
ax.scatter(seeds[:,0], seeds[:,1], s=3, c=np.clip(t,0,1), cmap=palette, vmin=0, vmax=1, lw=0)

ax = axs[1,2]; canvas(ax, '6. Final composite')
ax.add_collection(LineCollection(paths, colors=colors, linewidths=widths*0.4, capstyle='round'))

plt.tight_layout(); plt.show()

In [ ]:
# ---------- FINAL RENDER ----------
fig = plt.figure(figsize=(8, 10), dpi=300)
ax = fig.add_axes([0, 0, 1, 1])
fig.patch.set_facecolor(BG)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.25); ax.axis('off')
ax.add_collection(LineCollection(paths, colors=colors, linewidths=widths, capstyle='round'))
img = finish(fig_to_img(fig), name='flow_field.png', grain=3, vignette=0.25, display=True)